In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from PIL import Image

# ============================================================
# CONFIG
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Your generated X-ray image
#IMAGE_PATH = Path("../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/deb1f5bb-2026-03-03_15-40-15-784_te_000095_fake_B.png")
#IMAGE_PATH = Path("../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/9e487d1b-2026-03-03_16-54-56-681_te_000063_fake_B.png")
#IMAGE_PATH = Path("../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/1de1b63b-2026-03-02_16-10-48-409_te_000099_fake_B.png")
IMAGE_PATH = Path("../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/691989b9-2026-03-02_15-08-07-957_te_000089_fake_B.png")

# Your trained item-mask multi-head model
# The model was trained with 2 input channels, so we still provide a blank second channel internally.
#MODEL_PATH = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask/checkpoints/train_best.pt")
MODEL_PATH = Path("../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_itemmask_optional_slow/checkpoints/train_best_checkpoint.pt")

OUT_DIR = Path("../../reports/generated_image_eval/itemmask_multihead_same_as_normal_eval_blank_mask")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 512
GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

MASK_MEAN = (0.5,)
MASK_STD = (0.5,)

SPATIAL_CLASSES = ["isolated", "overlap"]          # 0, 1
THREAT_CLASSES = ["non_contraband", "contraband"] # 0, 1

GRADCAM_ALPHA = 0.40

# Same as normal evaluation: no extra cleanup before the transform.
# Set True only for debugging generated files with border-connected black padding.
APPLY_BORDER_CONNECTED_BLACK_CLEANUP = False
BORDER_BLACK_THRESHOLD = 15

print("Using device:", DEVICE)

# ============================================================
# MODEL
# Must match your item-mask training model exactly
# ============================================================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits

# ============================================================
# IMAGE HELPERS
# Same sizing logic as training validation/evaluation:
# resize while keeping aspect ratio, then pad to IMAGE_SIZE x IMAGE_SIZE.
# No zoom-in resize factor. No center crop.
# ============================================================

def resize_keep_ratio_and_pad(pil_img, target_size, interpolation, fill=255):
    """
    Resize image to fit inside target_size x target_size without cropping,
    then pad to target_size x target_size.

    For generated X-ray image:
      fill=255 gives white padding.

    For blank mask:
      fill=0 keeps the mask empty.
    """
    img = pil_img.convert("L")
    w, h = img.size

    scale = min(target_size / w, target_size / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    img = TF.resize(
        img,
        [new_h, new_w],
        interpolation=interpolation,
    )

    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2
    pad_right = target_size - new_w - pad_left
    pad_bottom = target_size - new_h - pad_top

    img = TF.pad(
        img,
        padding=[pad_left, pad_top, pad_right, pad_bottom],
        fill=fill,
    )

    return img

def remove_black_bands_keep_objects(pil_img, threshold=15, row_black_ratio=0.85, col_black_ratio=0.85):
    """
    Turns large black padding/bands into white, but tries not to remove dark objects.

    It detects rows/columns where most pixels are black.
    This is safer than changing all black pixels to white, because the blade/object
    may also contain dark pixels.
    """
    img = pil_img.convert("L")
    arr = np.array(img).astype(np.uint8)

    black = arr <= threshold
    h, w = black.shape

    # Detect mostly-black horizontal bands
    row_black_ratio_arr = black.mean(axis=1)
    black_rows = row_black_ratio_arr >= row_black_ratio

    # Detect mostly-black vertical bands
    col_black_ratio_arr = black.mean(axis=0)
    black_cols = col_black_ratio_arr >= col_black_ratio

    # Turn only those detected bands white
    arr[black_rows, :] = 255
    arr[:, black_cols] = 255

    return Image.fromarray(arr).convert("L")


def remove_border_connected_black(pil_img, threshold=15):
    """
    Only removes black padding connected to the image border.
    This is safer than turning ALL dark pixels white, because dark objects
    inside the tray should not be removed.
    """
    img = pil_img.convert("L")
    arr = np.array(img).astype(np.uint8)

    black = arr <= threshold
    h, w = black.shape

    visited = np.zeros_like(black, dtype=bool)
    stack = []

    # Start flood fill from border black pixels
    for x in range(w):
        if black[0, x]:
            stack.append((0, x))
        if black[h - 1, x]:
            stack.append((h - 1, x))

    for y in range(h):
        if black[y, 0]:
            stack.append((y, 0))
        if black[y, w - 1]:
            stack.append((y, w - 1))

    while stack:
        y, x = stack.pop()

        if y < 0 or y >= h or x < 0 or x >= w:
            continue

        if visited[y, x]:
            continue

        if not black[y, x]:
            continue

        visited[y, x] = True

        stack.append((y - 1, x))
        stack.append((y + 1, x))
        stack.append((y, x - 1))
        stack.append((y, x + 1))

    # Only border-connected black becomes white
    arr[visited] = 255

    return Image.fromarray(arr).convert("L")


def prepare_generated_image(image_path):
    """
    Loads generated image as grayscale.

    For same-as-normal evaluation, this function does NOT crop, zoom, or
    resize the image. The only model preprocessing happens later in the
    normal-evaluation-style transform: keep ratio + pad to 512 x 512.

    Optional border cleanup is disabled by default because normal evaluation
    does not apply extra cleanup before preprocessing.
    """
    img_raw = Image.open(image_path)
    print("Original image mode:", img_raw.mode)
    print("Original image size:", img_raw.size)

    img = img_raw.convert("L")

    # Remove large black bars/background from generated image
    img = remove_black_bands_keep_objects(
        img,
        threshold=15,
        row_black_ratio=0.85,
        col_black_ratio=0.85,
    )

    print("Removed large black padding/bands and changed them to white.")

    print("Processed image size before JointEvalTransform:", img.size)
    return img


def make_blank_mask(image_size):
    """
    The model expects 2 input channels because it was trained with image + item mask.
    For generated images without masks, use a blank second channel internally.
    This blank mask matches the no-polygon/blank-mask case in normal evaluation.
    """
    return Image.new("L", image_size, 0)


def joint_eval_transform_blank_mask(img):
    """
    Creates 2-channel input:
      channel 0 = generated grayscale X-ray
      channel 1 = blank mask

    This matches the normal evaluation transform:
      keep aspect ratio
      pad to IMAGE_SIZE x IMAGE_SIZE
      no center crop
      no zoom-in factor
    """
    blank_mask = make_blank_mask(img.size)

    img = resize_keep_ratio_and_pad(
        img,
        target_size=IMAGE_SIZE,
        interpolation=TF.InterpolationMode.BILINEAR,
        fill=255,
    )

    blank_mask = resize_keep_ratio_and_pad(
        blank_mask,
        target_size=IMAGE_SIZE,
        interpolation=TF.InterpolationMode.NEAREST,
        fill=0,
    )

    img_t = TF.to_tensor(img)
    mask_t = TF.to_tensor(blank_mask)

    img_t = TF.normalize(img_t, GRAY_MEAN, GRAY_STD)
    mask_t = TF.normalize(mask_t, MASK_MEAN, MASK_STD)

    x = torch.cat([img_t, mask_t], dim=0)
    return x


def denormalize_gray_channel(x_channel):
    arr = x_channel.detach().cpu().numpy()
    arr = arr * GRAY_STD[0] + GRAY_MEAN[0]
    arr = np.clip(arr, 0, 1)
    return arr


def tensor_to_display_rgb(x_tensor):
    """
    x_tensor: [1, 2, H, W]
    Uses channel 0 image only.
    """
    gray = denormalize_gray_channel(x_tensor[0, 0])
    img_u8 = (gray * 255).astype(np.uint8)
    img_rgb = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2RGB)
    return img_rgb


def make_heatmap_rgb(cam_01):
    heatmap_u8 = np.uint8(np.clip(cam_01, 0, 1) * 255)
    heatmap_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)
    return heatmap_rgb


def overlay_gradcam(display_rgb, cam_01, alpha=0.40):
    heatmap_rgb = make_heatmap_rgb(cam_01)
    overlay = cv2.addWeighted(display_rgb, 1 - alpha, heatmap_rgb, alpha, 0)
    return overlay

# ============================================================
# LOAD MODEL
# ============================================================
def safe_load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def clean_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state" in ckpt:
            state = ckpt["model_state"]
        elif "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        elif "state_dict" in ckpt:
            state = ckpt["state_dict"]
        else:
            state = ckpt
    else:
        state = ckpt

    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    return state


model = SimpleCNN_MultiHead(in_channels=2).to(DEVICE)

ckpt = safe_load_checkpoint(MODEL_PATH)
state = clean_state_dict(ckpt)

model.load_state_dict(state, strict=True)
model.eval()

print("Loaded model:", MODEL_PATH)

# ============================================================
# PREDICT GENERATED IMAGE
# ============================================================
img = prepare_generated_image(IMAGE_PATH)

x = joint_eval_transform_blank_mask(img).unsqueeze(0).to(DEVICE)

print("Input tensor shape:", tuple(x.shape))
print("Input tensor min/max:", x.min().item(), x.max().item())
print("Note: channel 1 is a blank mask used only for model compatibility.")

with torch.no_grad():
    spatial_logits, threat_logits = model(x)

    spatial_prob_t = torch.softmax(spatial_logits, dim=1)
    threat_prob_t = torch.softmax(threat_logits, dim=1)

    spatial_probs = spatial_prob_t.squeeze(0).cpu().numpy()
    threat_probs = threat_prob_t.squeeze(0).cpu().numpy()

    spatial_id = int(np.argmax(spatial_probs))
    threat_id = int(np.argmax(threat_probs))

spatial_pred = SPATIAL_CLASSES[spatial_id]
threat_pred = THREAT_CLASSES[threat_id]

contraband_prob = float(threat_probs[1])

# Same decision rule as normal evaluation: raw argmax over softmax probabilities.
threat_decision = threat_pred

print("\n===== GENERATED X-RAY MODEL PREDICTION =====")
print("Image:", IMAGE_PATH)

print("\nSpatial prediction:", spatial_pred, f"(class {spatial_id})")
for i, name in enumerate(SPATIAL_CLASSES):
    print(f"  {name}: {spatial_probs[i]:.4f}")

print("\nThreat raw prediction:", threat_pred, f"(class {threat_id})")
for i, name in enumerate(THREAT_CLASSES):
    print(f"  {name}: {threat_probs[i]:.4f}")

print("\nThreat decision:")
print(f"  final decision: {threat_decision}")

# Save prediction JSON
pred_json = OUT_DIR / "generated_xray_prediction_no_mask.json"
with open(pred_json, "w") as f:
    json.dump({
        "image_path": str(IMAGE_PATH),
        "model_path": str(MODEL_PATH),

        "spatial_prediction": spatial_pred,
        "spatial_class_id": spatial_id,
        "spatial_probabilities": {
            SPATIAL_CLASSES[i]: float(spatial_probs[i]) for i in range(len(SPATIAL_CLASSES))
        },

        "threat_raw_prediction": threat_pred,
        "threat_class_id": threat_id,
        "threat_probabilities": {
            THREAT_CLASSES[i]: float(threat_probs[i]) for i in range(len(THREAT_CLASSES))
        },

        "threat_decision": threat_decision,

        "input_channels": [
            "generated_grayscale_xray",
            "blank_mask_channel_for_model_compatibility"
        ],
        "mask_used": "blank_mask_channel",
        "image_size": IMAGE_SIZE,
        "resize_mode": "keep_aspect_ratio_pad_no_crop",
    }, f, indent=2)

print("\nSaved prediction JSON to:", pred_json)

# ============================================================
# DISPLAY INPUT ONLY — NO WHITE MATPLOTLIB BORDER
# ============================================================
display_rgb = tensor_to_display_rgb(x)
display_gray = denormalize_gray_channel(x[0, 0])

# Print prediction text instead of putting it inside the image canvas
print("\n===== DISPLAY IMAGE INFO =====")
print(f"Spatial: {spatial_pred}")
print(f"Threat raw: {threat_pred}")
print(f"Threat decision: {threat_decision}")

# Save clean input image with no matplotlib border/title
clean_input_path = OUT_DIR / "generated_xray_input_clean.png"
Image.fromarray(display_rgb).save(clean_input_path)
print("Saved clean input image to:", clean_input_path)

# Show image only, with white background
fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.imshow(display_gray, cmap="gray", vmin=0, vmax=1)
ax.axis("off")

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0)
plt.show()

# ============================================================
# GRAD-CAM
# ============================================================
class GradCAM:
    def __init__(self, model, target_layer, head_name="threat"):
        self.model = model
        self.target_layer = target_layer
        self.head_name = head_name

        self.activations = None
        self.gradients = None

        self.forward_handle = self.target_layer.register_forward_hook(self._save_activation)
        self.backward_handle = self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def __call__(self, x_tensor, class_idx=None):
        self.model.zero_grad(set_to_none=True)

        spatial_logits, threat_logits = self.model(x_tensor)

        if self.head_name == "spatial":
            logits = spatial_logits
            class_names = SPATIAL_CLASSES
        elif self.head_name == "threat":
            logits = threat_logits
            class_names = THREAT_CLASSES
        else:
            raise ValueError("head_name must be 'spatial' or 'threat'")

        probs = torch.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = int(torch.argmax(probs, dim=1).item())

        score = logits[:, int(class_idx)].sum()
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(
            cam,
            size=x_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        cam = cam.squeeze().detach().cpu().numpy()
        cam -= cam.min()
        cam /= (cam.max() + 1e-8)

        return cam


def get_conv_layers(model):
    layers = []

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            layers.append((name, module))

    return layers


conv_layers = get_conv_layers(model)

print("\nDetected Conv layers:")
for name, layer in conv_layers:
    print(" ", name)

# Last conv layer is usually the most useful high-level Grad-CAM layer
target_layer_name, target_layer = conv_layers[-1]



# ============================================================
# GRAD-CAM ALL LAYERS
# ============================================================
def run_gradcam_for_layer(head_name, target_layer_name, target_layer, target_class=None, suffix="predicted"):
    if head_name == "spatial":
        class_names = SPATIAL_CLASSES
        pred_id = spatial_id
    elif head_name == "threat":
        class_names = THREAT_CLASSES
        pred_id = threat_id
    else:
        raise ValueError("head_name must be 'spatial' or 'threat'")

    if target_class is None:
        target_class = pred_id

    gradcam = GradCAM(model, target_layer, head_name=head_name)
    cam = gradcam(x, class_idx=int(target_class))
    gradcam.remove_hooks()

    overlay = overlay_gradcam(display_rgb, cam, alpha=GRADCAM_ALPHA)
    heatmap = make_heatmap_rgb(cam)

    target_name = class_names[int(target_class)]
    safe_layer_name = target_layer_name.replace(".", "_").replace("[", "").replace("]", "")

    out_path = OUT_DIR / f"generated_xray_gradcam_{head_name}_{suffix}_{target_name}_{safe_layer_name}.png"
    Image.fromarray(overlay).save(out_path)

    heatmap_path = OUT_DIR / f"generated_xray_heatmap_{head_name}_{suffix}_{target_name}_{safe_layer_name}.png"
    Image.fromarray(heatmap).save(heatmap_path)

    return {
        "head_name": head_name,
        "target_class": int(target_class),
        "target_name": target_name,
        "layer_name": target_layer_name,
        "safe_layer_name": safe_layer_name,
        "cam": cam,
        "heatmap": heatmap,
        "overlay": overlay,
        "out_path": out_path,
    }


def run_gradcam_all_layers(head_name, target_class=None, suffix="predicted"):
    results = []

    for layer_name, layer in conv_layers:
        result = run_gradcam_for_layer(
            head_name=head_name,
            target_layer_name=layer_name,
            target_layer=layer,
            target_class=target_class,
            suffix=suffix,
        )
        results.append(result)

    return results


def show_gradcam_all_layers_grid(results, title):
    n = len(results)
    cols = 3
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(5 * cols, 5 * rows),
        facecolor="white"
    )

    axes = np.array(axes).reshape(-1)

    for i, result in enumerate(results):
        axes[i].imshow(result["overlay"])
        axes[i].set_title(
            f"{result['head_name'].upper()} | {result['target_name']}\n"
            f"{result['layer_name']}",
            color="black"
        )
        axes[i].axis("off")
        axes[i].set_facecolor("white")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
        axes[j].set_facecolor("white")

    fig.suptitle(title, fontsize=16, color="black")
    plt.tight_layout()
    plt.show()


def show_gradcam_all_layers_detailed(results, title_prefix):
    for result in results:
        fig, axes = plt.subplots(
            1,
            3,
            figsize=(15, 5),
            facecolor="white"
        )

        for ax in axes:
            ax.set_facecolor("white")

        axes[0].imshow(display_rgb)
        axes[0].set_title("Input generated X-ray", color="black")
        axes[0].axis("off")

        axes[1].imshow(result["heatmap"])
        axes[1].set_title(
            f"Heatmap\n"
            f"{result['head_name'].upper()} | {result['target_name']}\n"
            f"{result['layer_name']}",
            color="black"
        )
        axes[1].axis("off")

        axes[2].imshow(result["overlay"])
        axes[2].set_title(
            f"Overlay\n"
            f"{result['head_name'].upper()} | {result['target_name']}\n"
            f"{result['layer_name']}",
            color="black"
        )
        axes[2].axis("off")

        fig.suptitle(title_prefix, color="black")
        plt.tight_layout()
        plt.show()


# ============================================================
# RUN ALL-LAYER GRAD-CAM
# ============================================================
print("\n===== RUNNING ALL-LAYER GRAD-CAM =====")

# Spatial predicted class, all conv layers
spatial_gradcam_results = run_gradcam_all_layers(
    head_name="spatial",
    target_class=spatial_id,
    suffix="predicted",
)

show_gradcam_all_layers_grid(
    spatial_gradcam_results,
    title=f"Spatial Grad-CAM All Layers | Predicted: {spatial_pred}",
)

show_gradcam_all_layers_detailed(
    spatial_gradcam_results,
    title_prefix=f"Spatial Grad-CAM Detailed | Predicted: {spatial_pred}",
)

# Threat predicted class, all conv layers
threat_gradcam_results = run_gradcam_all_layers(
    head_name="threat",
    target_class=threat_id,
    suffix="predicted",
)

show_gradcam_all_layers_grid(
    threat_gradcam_results,
    title=f"Threat Grad-CAM All Layers | Predicted: {threat_pred}",
)

show_gradcam_all_layers_detailed(
    threat_gradcam_results,
    title_prefix=f"Threat Grad-CAM Detailed | Predicted: {threat_pred}",
)

# Also force contraband Grad-CAM if raw prediction is non_contraband
threat_contraband_gradcam_results = None

if threat_id != 1:
    print("\nRaw threat prediction is non_contraband.")
    print("Also showing forced contraband Grad-CAM for all layers.")

    threat_contraband_gradcam_results = run_gradcam_all_layers(
        head_name="threat",
        target_class=1,
        suffix="forced",
    )

    show_gradcam_all_layers_grid(
        threat_contraband_gradcam_results,
        title="Threat Grad-CAM All Layers | Forced Target: contraband",
    )

    show_gradcam_all_layers_detailed(
        threat_contraband_gradcam_results,
        title_prefix="Threat Grad-CAM Detailed | Forced Target: contraband",
    )

print("\n===== FINAL GENERATED X-RAY EVALUATION RESULT =====")
print("Image:", IMAGE_PATH)
print("Model:", MODEL_PATH)
print("Spatial:", spatial_pred, spatial_probs)
print("Threat raw:", threat_pred, threat_probs)
print("Threat decision:", threat_decision)
print("Prediction JSON:", pred_json)

print("\nSaved all-layer Grad-CAM overlays in:", OUT_DIR)

print("\nSpatial Grad-CAM files:")
for r in spatial_gradcam_results:
    print(" ", r["out_path"])

print("\nThreat predicted-class Grad-CAM files:")
for r in threat_gradcam_results:
    print(" ", r["out_path"])

if threat_contraband_gradcam_results is not None:
    print("\nThreat forced-contraband Grad-CAM files:")
    for r in threat_contraband_gradcam_results:
        print(" ", r["out_path"])